<a href="https://colab.research.google.com/github/ShahadSaeedAlshahrani/NLP_Labs/blob/main/Lab10/Lab10_Introduction_to_Machine_Translation_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Machine Translation (MT):

Machine Translation is a subfield of AI and linguistics that develops systems capable of translating text from one language to another. The primary goal of machine translation is to enable effective communication between people who speak different languages by automating the process of language translation.

Challenges in Machine Translation:
- Ambiguity

Ex. I went to the bank.
- Idiomatic Expressions

Ex. It's raining cats and dogs.
- Low-Resource Languages

Some languages have very limited data.

Applications of Machine Translation:
- Global Business and E-commerce
-Cross-Cultural Communication
- Multilingual Content Creation
- Language Learning



## Neural Machine Translation:

Neural Machine Translation is a data-driven approach where a single large neural network learns to represent both the source language and the target language, and to directly map between them.
Instead of using manually defined rules or phrase tables, it learns translation patterns automatically from large datasets of bilingual text.

### Key Components of Neural Machine Translation:



In [ ]:
%pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np

# Example bilingual dataset
english_sentences = ["Hello.", "How are you?", "This is a machine translation example."]
french_sentences = ["Bonjour.", "Comment ça va?", "Ceci est un exemple de traduction automatique."]

# Tokenize the english sentences
tokenizer_eng = Tokenizer()
tokenizer_eng.fit_on_texts(english_sentences)
# Tokenize words and assign an integer ID to each unique word
eng_sequences = tokenizer_eng.texts_to_sequences(english_sentences)
# Returns a list of tokens replaced by their index.
# Note: words index will start from 1
# because 0 is reserved by default for padding (not assigned to any word).
print(f"eng_sequences = {eng_sequences}")
eng_word_index = tokenizer_eng.word_index
# Returns the dictionary created by the tokenizer dict {word: index}.
print(f"eng_word_index = {eng_word_index}")

eng_sequences = [[1], [2, 3, 4], [5, 6, 7, 8, 9, 10]]
eng_word_index = {'hello': 1, 'how': 2, 'are': 3, 'you': 4, 'this': 5, 'is': 6, 'a': 7, 'machine': 8, 'translation': 9, 'example': 10}


In [ ]:
# Tokenize the french sentences
tokenizer_frn = Tokenizer()
tokenizer_frn.fit_on_texts(french_sentences)
frn_sequences = tokenizer_frn.texts_to_sequences(french_sentences)

print(f"frn_sequences = {frn_sequences}")
frn_word_index = tokenizer_frn.word_index
print(f"frn_word_index = {frn_word_index}")

frn_sequences = [[1], [2, 3, 4], [5, 6, 7, 8, 9, 10, 11]]
frn_word_index = {'bonjour': 1, 'comment': 2, 'ça': 3, 'va': 4, 'ceci': 5, 'est': 6, 'un': 7, 'exemple': 8, 'de': 9, 'traduction': 10, 'automatique': 11}


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Padding sequences
eng_padded = pad_sequences(eng_sequences)
# makes all sequences the same length by padding with zeros at the beginning (default)
print(f"eng_padded = {eng_padded}")

eng_padded = [[ 0  0  0  0  0  1]
 [ 0  0  0  2  3  4]
 [ 5  6  7  8  9 10]]


In [ ]:
frn_padded = pad_sequences(frn_sequences)
frn_padded

array([[ 0,  0,  0,  0,  0,  0,  1],
       [ 0,  0,  0,  0,  2,  3,  4],
       [ 5,  6,  7,  8,  9, 10, 11]], dtype=int32)

## LSTM Model



In [ ]:
from sklearn.model_selection import train_test_split

# Train-test split
eng_train, eng_test, frn_train, frn_test = train_test_split(eng_padded, frn_padded, test_size=0.2, random_state=42)

# Define the model
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(input_dim=len(eng_word_index) + 1, output_dim=64),
    tf.keras.layers.LSTM(64), # long short term memory
    tf.keras.layers.RepeatVector(frn_padded.shape[1]), #length of french sentence
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(frn_word_index) + 1, activation='softmax'))
])#convert to probabilities for each frn word

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# preparing the model's algorithms of optimizer, loss, metrics


# Train the model
model.fit(eng_train, np.expand_dims(frn_train, -1), epochs=50, batch_size=1, validation_split=0.2)


# Evaluate the model
loss, accuracy = model.evaluate(eng_test, np.expand_dims(frn_test, -1))
print(f'Test Accuracy: {accuracy}')

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0000e+00 - loss: 2.4833 - val_accuracy: 0.0000e+00 - val_loss: 2.4865
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.5714 - loss: 2.4738 - val_accuracy: 0.0000e+00 - val_loss: 2.4890
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.5714 - loss: 2.4637 - val_accuracy: 0.0000e+00 - val_loss: 2.4917
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.5714 - loss: 2.4527 - val_accuracy: 0.0000e+00 - val_loss: 2.4947
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step - accuracy: 0.5714 - loss: 2.4402 - val_accuracy: 0.0000e+00 - val_loss: 2.4981
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - accuracy: 0.5714 - loss: 2.4259 - val_accuracy: 0.0000e+00 - val_loss: 2.5019
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step - accuracy: 0.5714 - loss: 2.4091 - val_accuracy: 0.0000e+00 - val_loss: 2.5062
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - accuracy: 0.5714 - loss: 2.3893 - val_

```
# Define the model
model = tf.keras.Sequential([
```
This create a Sequential model,
which means that the layers are going to be implemented Sequentially.
And the each layer's output going to be input to the later layer

    `tf.keras.layers.Embedding(input_dim=len(eng_word_index) + 1, output_dim=64),
This defines the Embedding layer, the + 1 is for the padding,
64 is a Hyperparameter, which we can adjust as much as we like

    tf.keras.layers.LSTM(64), # long short term memory
THis is the LSTM **Encoder** layer, which will read the og Sentence (english) word for word  
and will revise the important info in the memory
64 -> indicate the LSTM Neural units.
**This layer Produces the context vector**

    tf.keras.layers.RepeatVector(frn_padded.shape[1]), #length of french sentence
a very Important layer  
this layer prepare for Decoding  
frn_padded.shape[1]--> this will equal the length of longest text in the sentence  
this layer will repeat the **Context Vector** *frn_padded.shape[1]* time
because The decoder requires a context vector at each time step
in order to generate each word of the target sentence.  


    tf.keras.layers.LSTM(64, return_sequences=True),
this is the decoding layer where the target sentence is generated  
return_sequences=True --> this returns the output for each time step  
64 -> indicate the LSTM Neural units.

    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(len(frn_word_index) + 1, activation='softmax'))
])
#convert to probabilities for each frn word

Dense layer convert the vector into scores in the target sentence  
softmax activation is converting these scores into Probabilities(Percentage).

in this example 64 is the features for each word
10 is the words length & the number of expected words in the target sentence

In [ ]:
# Inference
def translate_sentence(sentence, model, eng_tokenizer, frn_tokenizer):
    sequence = eng_tokenizer.texts_to_sequences([sentence])
    padded_sequence = pad_sequences(sequence, maxlen=eng_padded.shape[1])
    predicted_sequence = model.predict(padded_sequence) #outputs probabilities for each timestep (french padded sentence) and each unique word in the target vocabulary.
    translated_sequence = np.argmax(predicted_sequence, axis=-1) #Returns the index (word ID) with the largest value at each timestep.
    translated_sentence = frn_tokenizer.sequences_to_texts(translated_sequence)
    return translated_sentence[0]

# Example Inference
input_sentence = "How are you?"
translated_sentence = translate_sentence(input_sentence, model, tokenizer_eng, tokenizer_frn)
print(f'Input: {input_sentence}\nTranslation: {translated_sentence}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 267ms/step
Input: How are you?
Translation: comment ça ça


________________________________________________________________________________________________________________________________________

### Task 1: Understanding the Model Layers

Answer these simple questions in your own words:

1. What does the **Embedding** layer do to a word index?
3. What is the role of the **Encoder LSTM**?
5. What does the **Decoder LSTM** do?
8. Why do we use `softmax` in the final layer?


1) The Embedding layer converts each word index into a dense vector representing each word features so the model can learn meaningful relationships between words

2) The Encoder LSTM reads the whole English sentence and remembers its important meaning and save it into the Context vector

3) The Decoder LSTM uses the context vector to generate the output sequence step by step, and predect the translated words

4) We use softmax in the final layer to convert the outputs into probabilities for all possible French words, allowing the model to choose the most likely word at each time step.

### Task 2:

 Use Case: Language Translation (English-French)

A global e-commerce company wants to expand its reach to French-speaking customers and provide a seamless shopping experience. The company needs a language translation solution to automatically translate product listings, customer reviews, and other relevant content from English to French.

link: https://www.kaggle.com/datasets/devicharith/language-translation-englishfrench

In [ ]:
# import
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP_Lab10/eng_-french.csv")
df = df.head(5000)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,English words/sentences,French words/sentences
0,Hi.,Salut!
1,Run!,Cours !
2,Run!,Courez !
3,Who?,Qui ?
4,Wow!,Ça alors !


In [ ]:
english_sentences = df["English words/sentences"].astype(str)
french_sentences = df["French words/sentences"].astype(str)

In [ ]:
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)

eng_sequences = eng_tokenizer.texts_to_sequences(english_sentences)

eng_word_index = eng_tokenizer.word_index


In [ ]:
frn_tokenizer = Tokenizer()
frn_tokenizer.fit_on_texts(french_sentences)

frn_sequences = frn_tokenizer.texts_to_sequences(french_sentences)

frn_word_index = frn_tokenizer.word_index

In [ ]:
eng_padded = pad_sequences(eng_sequences)

frn_padded = pad_sequences(frn_sequences)

In [ ]:
print(eng_padded.shape)
print(frn_padded.shape)

(5000, 4)
(5000, 10)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(eng_padded, frn_padded,
    test_size=0.2, random_state=42)

In [ ]:
model = tf.keras.Sequential([
    # 1) Embedding
    tf.keras.layers.Embedding(input_dim=len(eng_word_index) + 1,
        output_dim=64),

    # 2) Encoder LSTM
    tf.keras.layers.LSTM(64),

    # 3) RepeatVector
    tf.keras.layers.RepeatVector(frn_padded.shape[1]),

    # 4) Decoder
    tf.keras.layers.LSTM(64, return_sequences=True),

    # 5) Predict French word probabilities
    tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(len(frn_word_index) + 1,activation='softmax'))
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
y_train = np.expand_dims(y_train, axis=-1)
y_test = np.expand_dims(y_test, axis=-1)
# because LSTM expects (samples, timesteps, 1)
# and y_train,y_test is (samples, timesteps)

In [ ]:
model.fit(X_train, y_train, validation_split=0.2, batch_size=64, epochs=10)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.6683 - loss: 6.0884 - val_accuracy: 0.6865 - val_loss: 3.2174
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6903 - loss: 2.7092 - val_accuracy: 0.6865 - val_loss: 2.7009
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6903 - loss: 2.6028 - val_accuracy: 0.6865 - val_loss: 2.7087
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6903 - loss: 2.5938 - val_accuracy: 0.6865 - val_loss: 2.7199
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6903 - loss: 2.5897 - val_accuracy: 0.6865 - val_loss: 2.7315
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6903 - loss: 2.5871 - val_accuracy: 0.6865 - val_loss: 2.7414
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6903 - loss: 2.5853 - val_accuracy: 0.6865 - val_loss: 2.7509
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6903 - loss: 2.5843 - val_accuracy: 0.6865 - v

In [ ]:
loss, acc = model.evaluate(X_test, y_test)

print("Accuracy:", acc)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6873 - loss: 2.7397
Accuracy: 0.6872999668121338


In [ ]:
input_sentence = "How are you?"
translated_sentence = translate_sentence(input_sentence, model, tokenizer_eng, tokenizer_frn)
print(f'Input: {input_sentence}\nTranslation: {translated_sentence}')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step
Input: How are you?
Translation: 
